In [ ]:
# heartfaliureprediction
# importing dependencies
import pandas as pd
from google.colab import drive
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from tensorflow.keras import callbacks

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data = pd.read_csv("./drive/MyDrive/Dataset/heart_failure.csv")

In [ ]:
# Scaling feature columns
scaling_req_list = ['creatinine_phosphokinase','ejection_fraction','platelets','serum_creatinine','serum_creatinine']
scaler = StandardScaler()
for column in scaling_req_list:
  data[column] = scaler.fit_transform(data[[column]])

In [ ]:
# Define the bin edges or intervals
bin_edges = [0, 30, 60, 90, 120, 180, 365]  # Example bin edges, adjust as per your requirements

# Create the labels for the bins
bin_labels = [6, 5, 4, 3, 2, 1]

# Bin the data based on the defined edges and labels
data['time'] = pd.cut(data['time'], bins=bin_edges, labels=bin_labels)


In [ ]:
data.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,0.000166,0,-1.530560,1,1.681648e-02,0.490057,130,1,0,6,1
1,55.0,0,7.514640,0,-0.007077,0,7.535660e-09,-0.284552,136,1,0,6,1
2,65.0,0,-0.449939,0,-1.530560,0,-1.038073e+00,-0.090900,129,1,1,6,1
3,50.0,1,-0.486071,0,-1.530560,0,-5.464741e-01,0.490057,137,1,0,6,1
4,65.0,1,-0.435486,1,-1.530560,0,6.517986e-01,1.264666,116,0,0,6,1


In [ ]:
# SPLITTING
feature = data.iloc[:,:-1]
target = data.iloc[:,-1]

In [ ]:
# train test split
x_train,x_test,y_train,y_test = train_test_split(feature,target,test_size=0.20)

In [ ]:
early_stopping = callbacks.EarlyStopping(
    min_delta=0.001, # minimium amount of change to count as an improvement
    patience=20, # how many epochs to wait before stopping
    restore_best_weights=True)

# creating model
model = Sequential(
    [
      layers.Dense(units=12,activation="relu",input_dim=12),
     layers.Dense(units=6,activation="relu"),
     layers.Dense(units=1,activation="sigmoid")
    ]
)

print(model.summary())

# Compiling ANN model
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])
history = model.fit(x_train,y_train,batch_size=16,epochs = 500,callbacks=[early_stopping],validation_split=0.2)

Model: "sequential_35"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_105 (Dense)           (None, 12)                156       
                                                                 
 dense_106 (Dense)           (None, 6)                 78        
                                                                 
 dense_107 (Dense)           (None, 1)                 7         
                                                                 
Total params: 241
Trainable params: 241
Non-trainable params: 0
_________________________________________________________________
None
Epoch 1/500
12/12 [==============================] - 1s 20ms/step - loss: 2.3770 - accuracy: 0.6859 - val_loss: 1.1943 - val_accuracy: 0.6458
Epoch 2/500
12/12 [==============================] - 0s 5ms/step - loss: 1.1023 - accuracy: 0.4136 - val_loss: 1.1691 - val_accuracy: 0.3333
Epoch 3/500
12/12 [============

In [ ]:
# model.weights

In [ ]:
# len(model.weights)

In [ ]:
# Comparing actual and predicted
ypred = model.predict(x_test)
ypred = (ypred > 0.4)

In [ ]:
print("Confusion Matrix : \n",confusion_matrix(y_true=y_test,y_pred=ypred))
print("Accuracy Score : ",accuracy_score(y_true=y_test,y_pred=ypred))
print("Classificaton Report : \n",classification_report(y_true=y_test,y_pred=ypred))


In [ ]:
history_df = pd.DataFrame(history.history)

import matplotlib.pyplot as plt

plt.plot(history_df.loc[:, ['loss']], "#6daa9f", label='Training loss')
plt.plot(history_df.loc[:, ['val_loss']],"#774571", label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc="best")

plt.show()

In [ ]:
history_df = pd.DataFrame(history.history)

plt.plot(history_df.loc[:, ['accuracy']], "#6daa9f", label='Training accuracy')
plt.plot(history_df.loc[:, ['val_accuracy']], "#774571", label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
